In [ ]:
"""
Análisis Tecno-Económico del Agregador en España
Cálculo y Normalización de la Tarifa Minorista (PVPC) y Excedentes

Este script procesa la serie temporal de la tarifa minorista regulada (PVPC)
obtenida a través de la API de ESIOS. Adicionalmente, cruza esta información con 
los precios marginales del Mercado Diario (DAM) para calcular el precio neto de 
compensación de excedentes, incorporando los costes operativos de desvío.

Autor: Alberto Zafra Muñoz
"""

import os
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional

# ==========================================
# 1. CONSTANTES Y CONFIGURACIÓN ECONÓMICA
# ==========================================
ARCHIVO_PVPC = "Tarifa_Demanda_PVPC_3_4_2026.csv"
ARCHIVO_DAM = "Precios_DAM_Limpios.csv"
ARCHIVO_SALIDA = "Tarifas_Usuario_Limpias.csv"
FECHA_ESTUDIO = '2026-04-03'

# Parámetros contractuales para la compensación de excedentes
PORCENTAJE_DAM = 1.00         # Indexación al 100% del mercado SPOT
COSTE_DESVIO_EUR_MWH = 2.50   # Penalización fija estimada por desvíos (€/MWh)

# ==========================================
# 2. PROCESAMIENTO DE DATOS (ETL)
# ==========================================
def procesar_tarifas_usuario(ruta_pvpc: str, ruta_dam: str) -> Optional[pd.DataFrame]:
    """
    Ingesta los datos de PVPC y DAM, limpia anomalías (duplicados de API) y 
    calcula la matriz económica de compra y venta de energía para el prosumidor.

    Parámetros:
        ruta_pvpc (str): Ruta al archivo CSV con los datos del PVPC.
        ruta_dam (str): Ruta al archivo CSV con los precios del mercado diario.

    Retorna:
        pd.DataFrame: DataFrame consolidado con tarifas de demanda e ingresos.
    """
    print("[*] Iniciando procesamiento de la matriz de tarifas minoristas...")
    
    if not os.path.exists(ruta_pvpc) or not os.path.exists(ruta_dam):
        print("[ERROR] No se encuentran los datasets fuente necesarios en el directorio.")
        return None

    try:
        # Carga de las series temporales
        df_pvpc = pd.read_csv(ruta_pvpc, sep=";")
        df_dam = pd.read_csv(ruta_dam)
        
        # 1. Limpieza y formateo del PVPC
        df_pvpc['datetime'] = pd.to_datetime(df_pvpc['datetime'])
        df_pvpc['Hora'] = df_pvpc['datetime'].dt.hour
        
        df_pvpc_clean = df_pvpc[['Hora', 'value']].rename(columns={'value': 'Precio_PVPC_EUR_MWh'})
        df_pvpc_clean = df_pvpc_clean.drop_duplicates(subset=['Hora']).reset_index(drop=True)
        
        # 2. Consolidación de datos (Merge)
        df_merged = pd.merge(df_pvpc_clean, df_dam, on='Hora', how='inner')
        
        if df_merged.empty:
            print("[ERROR] El cruce de datos (Merge) ha resultado en un DataFrame vacío.")
            return None

        # 3. Modelado Económico Contractual
        df_merged['Tarifa_Demanda_PVPC'] = df_merged['Precio_PVPC_EUR_MWh']
        
        # El ingreso por inyección a red descuenta los desvíos y se trunca en cero 
        # (evitando que el usuario pague por inyectar en horas de precios negativos)
        ingreso_bruto = (df_merged['Precio_DAM_EUR_MWh'] * PORCENTAJE_DAM) - COSTE_DESVIO_EUR_MWH
        df_merged['Tarifa_Ingreso_Excedentes'] = ingreso_bruto.clip(lower=0)
        
        print(f"[*] Matriz tarifaria calculada: {len(df_merged)} franjas horarias consolidadas.")
        return df_merged

    except Exception as e:
        print(f"[ERROR INESPERADO] Fallo durante la evaluación económica: {e}")
        return None

# ==========================================
# 3. REPRESENTACIÓN GRÁFICA ACADÉMICA
# ==========================================
def visualizar_tarifa_pvpc(df: pd.DataFrame, fecha: str):
    """
    Genera el gráfico del perfil de la tarifa minorista aplicable a los consumos.
    """
    plt.figure(figsize=(10, 5))
    plt.style.use('seaborn-v0_8-whitegrid')

    # Trazado del coste de adquisición de energía (Tarifa regulada)
    plt.plot(df['Hora'], df['Tarifa_Demanda_PVPC'], 
             color='#d62728', marker='o', linewidth=2.5, markersize=6, 
             label='Coste de Adquisición: Tarifa PVPC ($C_{u,t}^{DEM}$)')
    
    # Sombreado inferior para enfatizar el volumen económico
    plt.fill_between(df['Hora'], df['Tarifa_Demanda_PVPC'], 
                     color='#d62728', alpha=0.15)

    # Formateo y Estilos de la Figura
    plt.title(f'Evolución Horaria de la Tarifa Minorista de Compra (PVPC) - {fecha}', 
              fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Hora del día ($t$)', fontsize=12)
    plt.ylabel('Coste de la Energía (€/MWh)', fontsize=12)
    
    # Configuración de los ejes
    plt.xticks(range(0, 24))
    plt.xlim(0, 23)
    
    # Límite inferior forzado a 0 o mínimo real para contextualizar los valles
    y_min = min(0, df['Tarifa_Demanda_PVPC'].min() * 0.9)
    plt.ylim(y_min, df['Tarifa_Demanda_PVPC'].max() * 1.15)
    
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='upper left', frameon=True, shadow=True, fontsize=11)
    
    plt.tight_layout()
    plt.show()

# ==========================================
# 4. BLOQUE PRINCIPAL DE EJECUCIÓN
# ==========================================
if __name__ == "__main__":
    df_resultado = procesar_tarifas_usuario(ARCHIVO_PVPC, ARCHIVO_DAM)
    
    if df_resultado is not None:
        # Exportar el vector tarifario para uso en el optimizador MILP
        df_resultado.to_csv(ARCHIVO_SALIDA, index=False)
        print(f"[*] Perfil tarifario exportado satisfactoriamente a '{ARCHIVO_SALIDA}'.\n")
        
        # Mostrar matriz económica en consola
        pd.options.display.float_format = '{:.2f}'.format
        tabla_resumen = df_resultado[['Hora', 'Tarifa_Demanda_PVPC', 'Precio_DAM_EUR_MWh', 'Tarifa_Ingreso_Excedentes']]
        
        print("==================================================================")
        print(" VISTA PREVIA: CONDICIONES ECONÓMICAS DEL PROSUMIDOR (€/MWh)")
        print("==================================================================")
        print(tabla_resumen.to_string(index=False))
        print("==================================================================\n")
        
        print("[*] Generando renderizado visual del PVPC...")
        visualizar_tarifa_pvpc(df_resultado, FECHA_ESTUDIO)